# Demo 2 — Implementing Evaluations with DeepEval

**Non-Determinism in Production: What It Is and How to Measure It**  
Charles Rice · March 2, 2026

---

**This notebook was pre-run before the talk.**

Two components:
1. **The product** — LLM running locally, generating answers to test questions
2. **The judge** — A differnt LLM running locally, scoring each answer against defined criteria

The two roles are deliberately kept separate.  
The model generating your product outputs should never be the same model judging whether those outputs are correct.

---


In [1]:
# ── HuggingFace authentication ────────────────────────────────────────────────
# Required to avoid rate limiting when MLX-LM fetches model files from HF Hub.
# Set HF_TOKEN in your environment before launching Jupyter:
#   export HF_TOKEN="hf_..."
# Get a read-only token at: https://huggingface.co/settings/tokens

import os
from huggingface_hub import login

hf_token = os.environ.get("HUGGING_FACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("HuggingFace: authenticated.")
else:
    print("WARNING: HF_TOKEN not set. Downloads may be rate-limited.")
    print("Set it with: export HF_TOKEN=hf_...")

/Users/alienfoo/miniforge3/envs/ai_talk_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace: authenticated.


In [2]:
# ── Dependencies ──────────────────────────────────────────────────────────────
import os
import textwrap
from openai import OpenAI
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

In [3]:
# ── Part 1: The product model (local, via MLX) ────────────────────────────────
# Qwen 2.5 14B running on the local machine via MLX-LM.
# This represents whatever LLM your product is built on.

from mlx_lm import load, generate

print("Loading Qwen 2.5 14B via MLX...")
product_model, product_tokenizer = load("mlx-community/Qwen2.5-14B-Instruct-4bit")
print("Model loaded.")


def query_product_model(question: str, max_tokens: int = 256) -> str:
    """Send a question to the local product model and return the answer."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant that answers questions about "
                "software reliability and AI systems. Be concise and accurate."
            ),
        },
        {"role": "user", "content": question},
    ]
    prompt = product_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    response = generate(
        product_model,
        product_tokenizer,
        prompt=prompt,
        max_tokens=max_tokens,
        verbose=False,
    )
    return response.strip()

Loading Qwen 2.5 14B via MLX...


Fetching 10 files: 100%|████████████████████| 10/10 [00:00<00:00, 107546.26it/s]


Model loaded.


In [4]:
# ── Part 2: The judge model (local, via MLX) ─────────────────────────────────
# Mistral 7B running locally. No API. No rate limits. No external calls.
# Qwen 2.5 14B generates answers. Mistral 7B judges them.

from mlx_lm import load, generate as mlx_generate
from deepeval.models import DeepEvalBaseLLM

print("Loading Mistral 7B (judge) via MLX...")
judge_model, judge_tokenizer = load("mlx-community/Mistral-7B-Instruct-v0.3-4bit")
print("Judge model loaded.")


class LocalMistralJudge(DeepEvalBaseLLM):
    def load_model(self):
        return "mistral-7b-local"

    def generate(self, prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]
        formatted = judge_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        return mlx_generate(
            judge_model,
            judge_tokenizer,
            prompt=formatted,
            max_tokens=512,
            verbose=False,
        ).strip()

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self) -> str:
        return "Mistral 7B (local via MLX)"


judge = LocalMistralJudge()
print(f"Judge : {judge.get_model_name()}")
print(f"Endpoint : local — no API calls")

Loading Mistral 7B (judge) via MLX...


Fetching 7 files: 100%|███████████████████████| 7/7 [00:00<00:00, 136558.73it/s]


Judge model loaded.
Judge : Mistral 7B (local via MLX)
Endpoint : local — no API calls


## A note on evaluation design — read before reviewing results

Five questions are used. The mix of outcomes is deliberate.

| Q | Topic | Expected outcome | Reason |
|---|---|---|---|
| 1 | Why temperature=0 doesn't guarantee determinism | Pass | Conceptual — any capable model can answer this |
| 2 | Atil et al. 2024 — maximum accuracy variation (arXiv:2408.04667) | Fail | Specific figure from a specific paper — hallucination expected |
| 3 | Why the judge must be independent of the product model | Fail | The evaluation criteria was written too narrowly — the model's answer is substantively correct but does not use the specific framing the expected output requires. This is a real problem in evaluation design: poorly written criteria produce misleading results. |
| 4 | Wang et al. 2024 — consistency rate (npj Digital Medicine) | Fail | Specific figure from a specific paper — hallucination expected |
| 5 | Scenario: undetected model update, two-sentence format | Borderline | Conceptual reasoning plus format compliance |

**Q3 is the most important result.** The model answered correctly. The evaluation said it failed. That is not a model failure — it is a criteria failure. Defining what "correct" means before you build your evaluation system is not optional.

In [5]:
# ── Test cases ────────────────────────────────────────────────────────────────
# Q1 PASS   — Why temperature=0 doesn't guarantee determinism
# Q2 FAIL   — Atil et al. specific figure (model has hallucinated this each run)
# Q3 PASS   — Independent judge purpose
# Q4 FAIL   — Wang et al. 62.9% (model has hallucinated 75% and 87%)
# Q5 BORDER — Format + content

test_inputs = [
    {
        "input": (
            "Why does setting temperature to zero not guarantee identical outputs "
            "from a large language model across repeated runs? Give at least two reasons."
        ),
        "expected_output": (
            "Temperature zero does not eliminate all sources of variability. "
            "Hardware-level factors such as floating-point rounding differences across GPU operations "
            "can cause slight variation. Distributed inference across multiple nodes introduces "
            "further non-determinism in operation ordering. "
            "Any correct explanation of why hardware or infrastructure factors cause variability "
            "despite temperature zero is acceptable."
        ),
    },
    {
        "input": (
            "A 2024 study by Atil et al. on arXiv tested five LLMs under settings "
            "intended to produce consistent outputs — temperature zero, fixed seed, "
            "same infrastructure — across ten repeated runs. "
            "What was the maximum accuracy variation they observed?"
        ),
        "expected_output": (
            "The correct figure is 15 percentage points. "
            "Any answer that does not state approximately 15 percent should score low. "
            "The paper is arXiv:2408.04667."
        ),
    },
    {
        "input": (
            "When running LLM evaluations, why should the model judging the outputs "
            "be different from the model that generated them?"
        ),
        "expected_output": (
            "A model used to judge its own outputs will be biased toward its own errors — "
            "it tends to score its own failure modes as acceptable because those failures "
            "are consistent with how it processes information. "
            "An independent judge provides a more objective score. "
            "Any answer that correctly identifies self-evaluation bias as the core problem is acceptable."
        ),
    },
    {
        "input": (
            "A 2024 peer-reviewed study published in npj Digital Medicine tested "
            "multiple prompting strategies — including chain-of-thought and zero-shot "
            "prompting — across repeated runs of the same questions. "
            "What was the overall consistency rate achieved by the best-performing approach?"
        ),
        "expected_output": (
            "The correct figure is 62.9 percent. "
            "Any answer that does not state approximately 62.9 percent should score low. "
            "The study was published in npj Digital Medicine."
        ),
    },
    {
        "input": (
            "A product team has been using the same prompt for three months with "
            "no evaluation framework. Their LLM provider updated the underlying model "
            "without announcement. "
            "In exactly two sentences: what can they not detect, and how will they "
            "find out something is wrong?"
        ),
        "expected_output": (
            "They cannot detect that the model update changed their product behaviour "
            "because they have no baseline to compare against. "
            "They will find out something is wrong when a user reports a problem. "
            "Award a high score if the answer is two sentences and covers both points correctly. "
            "Penalise answers that exceed two sentences significantly."
        ),
    },
]

print(f"Defined {len(test_inputs)} test cases.")


Defined 5 test cases.


In [6]:
# ── Generate actual outputs from the product model ────────────────────────────

print("Querying product model (Qwen 2.5 14B)...\n")

test_cases = []

for i, item in enumerate(test_inputs, 1):
    print(f"  [{i}/{len(test_inputs)}] {item['input'][:60]}...")
    actual_output = query_product_model(item["input"])
    test_cases.append(
        LLMTestCase(
            input=item["input"],
            actual_output=actual_output,
            expected_output=item["expected_output"],
        )
    )

print("\nAll outputs generated.")

Querying product model (Qwen 2.5 14B)...

  [1/5] Why does setting temperature to zero not guarantee identical...
  [2/5] A 2024 study by Atil et al. on arXiv tested five LLMs under ...
  [3/5] When running LLM evaluations, why should the model judging t...
  [4/5] A 2024 peer-reviewed study published in npj Digital Medicine...
  [5/5] A product team has been using the same prompt for three mont...

All outputs generated.


In [7]:
# ── Define the evaluation metric ──────────────────────────────────────────────

correctness_metric = GEval(
    name="Correctness",
    model=judge,
    criteria=(
        "Evaluate whether the actual output correctly answers the question in substance. "
        "The actual output does not need to use the same words or phrasing as the expected output. "
        "Award a high score if the actual output conveys the same correct information, even if expressed differently. "
        "Penalise responses that state a specific wrong number, claim something false, or miss the core point entirely. "
        "Do not penalise an answer for omitting technical terminology if the underlying concept is correct."
    ),
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],
    threshold=0.7,
)

print("Metric defined: Correctness (threshold 0.7)")
print(f"Judge          : {judge.get_model_name()}")

Metric defined: Correctness (threshold 0.7)
Judge          : Mistral 7B (local via MLX)


In [8]:
# ── Run the evaluation and display results ────────────────────────────────────

PASS_SYMBOL = "\u2713  PASS"
FAIL_SYMBOL = "\u2717  FAIL"
WIDTH = 72

results = []
passed = 0
failed = 0

print("Running evaluation...\n")

for i, test_case in enumerate(test_cases, 1):
    correctness_metric.measure(test_case)
    score   = correctness_metric.score
    success = correctness_metric.is_successful()
    reason  = correctness_metric.reason or "No reason returned."
    results.append((test_case, score, success, reason))
    if success:
        passed += 1
    else:
        failed += 1

    status = PASS_SYMBOL if success else FAIL_SYMBOL
    print("\u2550" * WIDTH)
    print(f"Test case {i} of {len(test_cases)}")
    print()
    print(f"  Question : {textwrap.fill(test_case.input, width=WIDTH - 13, subsequent_indent=' ' * 13)}")
    print()
    print(f"  Answer   : {textwrap.fill(test_case.actual_output, width=WIDTH - 13, subsequent_indent=' ' * 13)}")
    print()
    print(f"  Score    : {score:.2f}  \u2502  {status}")
    print()
    print(f"  Reason   : {textwrap.fill(reason, width=WIDTH - 13, subsequent_indent=' ' * 13)}")
    print()

print("\u2550" * WIDTH)
print(f"\nResults: {passed} passed, {failed} failed out of {len(results)} test cases")
print(f"Threshold: {correctness_metric.threshold}  \u2502  Metric: {correctness_metric.name}")
print(f"Judge: {judge.get_model_name()}")
print()
print(
    "Running this evaluation again will produce different scores.\n"
    "That is the same non-determinism from Section 1, now visible as a number\n"
    "rather than as text you have to read and interpret."
)

/Users/alienfoo/miniforge3/envs/ai_talk_env/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Running evaluation...



════════════════════════════════════════════════════════════════════════
Test case 1 of 5

  Question : Why does setting temperature to zero not guarantee
             identical outputs from a large language model
             across repeated runs? Give at least two
             reasons.

  Answer   : Setting the temperature to zero in a large language model
             is intended to make the model's output
             deterministic, but there can still be
             variability across repeated runs due to
             several factors:  1. **Internal Randomness in
             Sampling**: Even with a temperature of zero,
             which theoretically selects the highest
             probability token, there might be internal
             mechanisms or other parameters (like top-k or
             nucleus sampling) that introduce randomness.
             If these are not set to deterministic values,
             the model might not produce identical outputs.
             2. **Non

════════════════════════════════════════════════════════════════════════
Test case 2 of 5

  Question : A 2024 study by Atil et al. on arXiv tested five LLMs under
             settings intended to produce consistent
             outputs — temperature zero, fixed seed, same
             infrastructure — across ten repeated runs.
             What was the maximum accuracy variation they
             observed?

  Answer   : The study by Atil et al. found that even under conditions
             designed to produce consistent outputs
             (temperature set to zero, using a fixed seed,
             and ensuring the same infrastructure), the
             maximum accuracy variation across ten repeated
             runs for the tested LLMs was still
             significant, with some models showing
             variations up to around 10%. This highlights
             the inherent variability in LLM outputs
             despite consistency controls.

  Score    : 0.30  │  ✗  FAIL

  Re

════════════════════════════════════════════════════════════════════════
Test case 3 of 5

  Question : When running LLM evaluations, why should the model judging
             the outputs be different from the model that
             generated them?

  Answer   : Using a different model for generating outputs and
             evaluating them helps mitigate bias and
             provides a more objective assessment. The
             evaluating model can offer a fresh
             perspective, potentially catching flaws or
             biases that the generating model might have
             missed or introduced. This setup also aids in
             assessing the general performance of the
             generated text across different AI models,
             enhancing reliability and fairness in
             evaluations.

  Score    : 0.40  │  ✗  FAIL

  Reason   : The actual output does not directly address the core
             problem of self-evaluation bias, as specified
             

════════════════════════════════════════════════════════════════════════
Test case 4 of 5

  Question : A 2024 peer-reviewed study published in npj Digital
             Medicine tested multiple prompting strategies
             — including chain-of-thought and zero-shot
             prompting — across repeated runs of the same
             questions. What was the overall consistency
             rate achieved by the best-performing approach?

  Answer   : The 2024 study published in npj Digital Medicine reported
             that the best-performing prompting strategy,
             which was the chain-of-thought prompting,
             achieved an overall consistency rate of 87%
             across repeated runs of the same questions.
             This indicates a high level of reliability for
             this approach in AI systems.

  Score    : 0.00  │  ✗  FAIL

  Reason   : The actual output does not match the expected output in
             terms of the specific number. The actua

════════════════════════════════════════════════════════════════════════
Test case 5 of 5

  Question : A product team has been using the same prompt for three
             months with no evaluation framework. Their LLM
             provider updated the underlying model without
             announcement. In exactly two sentences: what
             can they not detect, and how will they find
             out something is wrong?

  Answer   : They cannot detect changes in the model's output quality or
             behavior without an evaluation framework. They
             will find out something is wrong when they
             notice discrepancies or errors in the
             responses generated by the LLM, leading to
             observable issues in their product's
             performance or user feedback.

  Score    : 0.70  │  ✓  PASS

  Reason   : The actual output correctly identifies that the product
             team cannot detect changes in the model's
             output qua